# Milestone 1: Data Profiling

Notebook ini melakukan inspeksi dan *profiling* awal terhadap **Indonesian News Dataset** (berita dari Tempo, CNN Indonesia, CNBC Indonesia, Okezone, Suara, Kumparan, dan JawaPos) secara efisien menggunakan **Polars** dengan *lazy evaluation*.

Sumber data: https://www.kaggle.com/datasets/iqbalmaulana/indonesian-news-dataset

## 0. Konfigurasi dan Import

In [1]:
%pip install polars

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import polars as pl

# Tambahkan huruf r di depan string path (raw string) agar backslash (\) tidak dibaca sebagai escape character
DATA_PATH = r"D:\1. UMM\SEMESTER 7\ANALISIS BIG DATA\data.csv"

# Mengatur tampilan output Polars
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_tbl_rows(20)

print("File ditemukan:", os.path.exists(DATA_PATH))

File ditemukan: True


## 1. Memuat Dataset (Lazy)

Dataset berisi teks artikel yang panjang dan kolom `embedding` berupa vektor dalam bentuk teks. Semua kolom dibaca sebagai `String` terlebih dahulu (`infer_schema_length=0`) agar pembacaan tidak gagal karena tipe yang salah tebak. Konversi tipe dilakukan setelahnya.

In [3]:
lf = pl.scan_csv(DATA_PATH, infer_schema_length=0)

# Kolom 'embedding' sangat besar dan tidak dipakai untuk profiling,
# jadi dibuang sejak awal agar query lebih ringan.
lf = lf.drop("embedding")

columns = lf.collect_schema().names()
print("Kolom:", columns)

Kolom: ['id', 'source', 'title', 'image', 'url', 'content', 'date', 'created_at', 'updated_at', 'summary']


## 2. Informasi Ukuran Dataset

Syarat tugas: ukuran file ≥ 500 MB **atau** lebih dari 1 juta baris.

In [4]:
size_mb = os.path.getsize(DATA_PATH) / (1024 ** 2)
total_rows = lf.select(pl.len()).collect().item()

print(f"Ukuran file : {size_mb:,.1f} MB")
print(f"Total baris : {total_rows:,}")
print(f"Total kolom : {len(columns)} (di luar 'embedding')")
print("Memenuhi syarat ukuran (>= 500 MB) :", size_mb >= 500)
print("Memenuhi syarat baris (> 1 juta)   :", total_rows > 1_000_000)

Ukuran file : 703.7 MB
Total baris : 32,735
Total kolom : 10 (di luar 'embedding')
Memenuhi syarat ukuran (>= 500 MB) : True
Memenuhi syarat baris (> 1 juta)   : False


## 3. Mengintip Sampel Data (Head)

In [5]:
lf.select(["id", "source", "title", "date", "created_at"]).head(5).collect()

id,source,title,date,created_at
str,str,str,str,str
"""83""","""tempo""","""Depo Plumpang Terbakar, Anggota DPR Minta Pertamina Pastikan…","""2023-03-04 06:18:13+00""","""2023-03-04 07:03:39.039332"""
"""84""","""tempo""","""Jokowi Perintahkan Wapres Ma'ruf Amin Tinjau Lokasi Kebakara…","""2023-03-04 06:04:38+00""","""2023-03-04 07:03:39.039332"""
"""85""","""tempo""","""HNW Mendukung Jamaah Umroh First Travel Dapatkan Haknya""","""2023-03-04 06:18:04+00""","""2023-03-04 07:03:39.039332"""
"""86""","""tempo""","""Tim Dokkes Polri Telah Terima 14 Kantong Jenazah Korban Keba…","""2023-03-04 06:44:10+00""","""2023-03-04 07:03:39.039332"""
"""87""","""tempo""","""Bamsoet Ajak Komunitas Otomotif Kembangkan Perekonomian Nasi…","""2023-03-04 06:38:57+00""","""2023-03-04 07:03:39.039332"""


## 4. Pengecekan Nilai Kosong (Null dan String Kosong)

Karena semua kolom dibaca sebagai `String`, nilai kosong bisa berupa `null` **atau** teks kosong/spasi. Keduanya dihitung.

In [6]:
null_report = lf.select([
    pl.col(c).is_null().sum().alias(f"{c}__null") for c in columns
] + [
    (pl.col(c).str.strip_chars() == "").sum().alias(f"{c}__kosong") for c in columns
]).collect()

report = pl.DataFrame({
    "kolom": columns,
    "null": [null_report[f"{c}__null"][0] for c in columns],
    "string_kosong": [null_report[f"{c}__kosong"][0] for c in columns],
}).with_columns(
    ((pl.col("null") + pl.col("string_kosong")) / total_rows * 100).round(2).alias("persen_kosong")
)
report

kolom,null,string_kosong,persen_kosong
str,i64,i64,f64
"""id""",0,0,0.0
"""source""",0,0,0.0
"""title""",0,0,0.0
"""image""",0,160,0.49
"""url""",0,0,0.0
"""content""",0,441,1.35
"""date""",0,0,0.0
"""created_at""",0,0,0.0
"""updated_at""",0,0,0.0


## 5. Statistik Deskriptif Dasar

### 5.1 Panjang teks (judul, isi, ringkasan)

In [7]:
lengths = lf.select([
    pl.col("title").str.len_chars().alias("panjang_judul"),
    pl.col("content").str.len_chars().alias("panjang_isi"),
    pl.col("summary").str.len_chars().alias("panjang_ringkasan"),
]).collect()

lengths.describe()

statistic,panjang_judul,panjang_isi,panjang_ringkasan
str,f64,f64,f64
"""count""",32735.0,32735.0,32725.0
"""null_count""",0.0,0.0,10.0
"""mean""",67.359249,2266.797831,446.038105
"""std""",14.514869,1212.222629,116.242655
"""min""",16.0,0.0,59.0
"""25%""",58.0,1515.0,368.0
"""50%""",66.0,2070.0,451.0
"""75%""",76.0,2775.0,525.0
"""max""",142.0,13833.0,5724.0


### 5.2 Sebaran artikel per sumber berita

In [8]:
per_source = (
    lf.group_by("source")
    .agg(pl.len().alias("jumlah_artikel"))
    .sort("jumlah_artikel", descending=True)
    .collect()
    .with_columns((pl.col("jumlah_artikel") / total_rows * 100).round(2).alias("persen"))
)
per_source

source,jumlah_artikel,persen
str,u32,f64
"""kumparan""",8579,26.21
"""okezone""",7066,21.59
"""cnbcindonesia""",4691,14.33
"""suara""",4004,12.23
"""cnnindonesia""",3738,11.42
"""tempo""",3642,11.13
"""jawapos""",1015,3.1


## 6. Profil Waktu (Kolom `date`)

Kolom `date` dikonversi ke tipe datetime. Nilai yang gagal dikonversi menjadi `null` (`strict=False`) dan dihitung agar kualitasnya terdokumentasi.

In [9]:
# Lihat dulu contoh isi kolom date
lf.select("date").head(5).collect()

date
str
"""2023-03-04 06:18:13+00"""
"""2023-03-04 06:04:38+00"""
"""2023-03-04 06:18:04+00"""
"""2023-03-04 06:44:10+00"""
"""2023-03-04 06:38:57+00"""


In [10]:
dated = lf.select(
    pl.col("date"),
    pl.col("date").str.slice(0, 10).str.to_date("%Y-%m-%d", strict=False).alias("date_parsed"),
).collect()

gagal = dated.filter(pl.col("date_parsed").is_null()).height
print(f"Tanggal gagal di-parse : {gagal:,} dari {total_rows:,}")
print("Tanggal paling awal   :", dated["date_parsed"].min())
print("Tanggal paling akhir  :", dated["date_parsed"].max())

Tanggal gagal di-parse : 0 dari 32,735
Tanggal paling awal   : 2018-08-10
Tanggal paling akhir  : 2023-04-11


In [11]:
per_month = (
    dated.drop_nulls("date_parsed")
    .group_by(pl.col("date_parsed").dt.truncate("1mo").alias("bulan"))
    .agg(pl.len().alias("jumlah_artikel"))
    .sort("bulan")
)
per_month

bulan,jumlah_artikel
date,u32
2018-08-01,1
2023-03-01,21223
2023-04-01,11511


## 7. Pengecekan Duplikasi Awal

Hanya pengecekan awal. Penanganan duplikasi dilakukan di Milestone 2 (cleaning).

In [12]:
dup = lf.select(
    pl.col("id").n_unique().alias("id_unik"),
    pl.col("url").n_unique().alias("url_unik"),
    pl.col("title").n_unique().alias("judul_unik"),
).collect()

dup = dup.with_columns(
    (total_rows - pl.col("id_unik")).alias("duplikat_id"),
    (total_rows - pl.col("url_unik")).alias("duplikat_url"),
    (total_rows - pl.col("judul_unik")).alias("duplikat_judul"),
)
dup

id_unik,url_unik,judul_unik,duplikat_id,duplikat_url,duplikat_judul
u32,u32,u32,u32,u32,u32
32735,32735,32651,0,0,84
